# Problema de la mochila algorítmo genético

## Importaciones

In [15]:
import numpy as np

rng = np.random.default_rng()

## Funciones a definir

In [21]:
def generacion(tamanio_array,n_inicial):
    soluciones = rng.integers(0,1,size = (n_inicial,tamanio_array))
    return soluciones

In [22]:
def torneo(solucion1,solucion2,pesos,utilidades,peso_max):

    # Calculamos los pesos de las soluciones
    peso_solucion1 = pesos[solucion1 > 0.5].sum()
    peso_solucion2 = pesos[solucion2 > 0.5].sum()

    # Calculamos las utilidades de las soluciones
    if peso_solucion1 > peso_max:
        utilidad_solucion1 = 0
    else:
        utilidad_solucion1 = utilidades[solucion1>0.5].sum()


    if peso_solucion2 > peso_max:
        utilidad_solucion2 = 0
    else:
        utilidad_solucion2 = utilidades[solucion2>0.5].sum()

    # Elegimos ganador
    if utilidad_solucion1 > utilidad_solucion2:
        ganador = solucion1
    elif utilidad_solucion2 > utilidad_solucion1:
        ganador = solucion2
    else:
        # En caso de empate elegimos uno al azar
        ganador = solucion1 if np.random.rand() < 0.5 else solucion2

    return ganador

In [23]:
def seleccion(soluciones,pesos,utilidades,peso_max):
    padres = []

    while len(padres) < 2:
        indices = rng.integers(len(soluciones),size = 2)
        solucion1 = soluciones[indices[0]]
        solucion2 = soluciones[indices[1]]

        ganador = torneo(solucion1,solucion2,pesos,utilidades,peso_max)

        padres.append(ganador)

    return padres[0],padres[1]

In [39]:
def cruce(padre1,padre2):

    # hacemos el corte
    corte = rng.integers(1,len(padre1))

    # aplicamos el corte
    padre1_superior = padre1[:corte]
    padre1_inferior = padre1[corte:]
    padre2_superior = padre2[:corte]
    padre2_inferior = padre2[corte:]

    # cruzamos para tener hijos
    hijo1 = np.concatenate((padre1_superior, padre2_inferior))
    hijo2 = np.concatenate((padre2_superior, padre1_inferior))

    return hijo1, hijo2


In [40]:
def mutacion(solucion, porcentaje_mutacion):
    for i in range(len(solucion)):
        if rng.random() < porcentaje_mutacion:
            solucion[i] = 1 - solucion[i]

    return solucion

In [41]:
def mejor_solucion(soluciones,pesos,utilidades,peso_max):
    
    maximo = soluciones[0]
    utilidad_maxima = -1e4
    
    for solucion in soluciones:
        peso_solucion = pesos[solucion > 0.5].sum()

        if peso_solucion > peso_max:
            utilidad_solucion = 0
        else:
            utilidad_solucion = utilidades[solucion>0.5].sum()
        if utilidad_solucion > utilidad_maxima:
            maximo = solucion
            utilidad_maxima = utilidad_solucion
            
    return maximo

## Condiciones del problema

In [42]:
objetos = ["Agua", "Comida", "Abrigo", "Botiquín", "Linterna", "Tienda de campaña", "Cámara", "Libro"]

pesos =np.array( [3, 2, 2, 1, 1, 5, 2, 1])

utilidades = np.array([10, 9, 7, 8, 5, 6, 4, 2])

peso_max = 10

porcentaje_mutacion = 0.05

tamanio_array = 8

n_inicial = 20

n_generaciones = 100

## Main del problema

In [43]:
# Generamos la solucion incial

soluciones = generacion(tamanio_array,n_inicial)

mejores = []

for i in range(n_generaciones):
    
    siguiente_generacion = []

    mejores.append(mejor_solucion(soluciones,pesos,utilidades,peso_max))
    
    siguiente_generacion.append(mejor_solucion(soluciones,pesos,utilidades,peso_max))

    while  len(siguiente_generacion) < 20:
        padre1,padre2 = seleccion(soluciones,pesos,utilidades,peso_max)
    
        hijo1,hijo2 = cruce(padre1,padre2)

        hijo1 = mutacion(hijo1,porcentaje_mutacion)
        hijo2 = mutacion(hijo2,porcentaje_mutacion)

        if len(siguiente_generacion) < 19:
            siguiente_generacion.append(hijo1)
            siguiente_generacion.append(hijo2)

        else:
            siguiente_generacion.append(torneo(hijo1,hijo2,pesos,utilidades,peso_max))

    soluciones = siguiente_generacion

    

In [46]:
mejores[:11]

[array([0, 0, 0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 1, 0, 0, 0, 0]),
 array([1, 0, 0, 0, 0, 0, 0, 0]),
 array([1, 0, 0, 1, 0, 0, 0, 0]),
 array([1, 0, 1, 1, 0, 0, 0, 1]),
 array([1, 0, 1, 1, 0, 0, 0, 1]),
 array([1, 0, 1, 1, 0, 0, 1, 0]),
 array([1, 1, 0, 1, 1, 0, 0, 0]),
 array([1, 1, 0, 1, 1, 0, 1, 0]),
 array([1, 1, 1, 1, 0, 0, 1, 0]),
 array([1, 1, 1, 1, 1, 0, 0, 1])]